# Demo 7 — Durata osservata e griglia della FFT

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
for durata in (0.005, 0.02, 0.1):
    n = int(durata * SR)
    x = np.sin(2 * np.pi * 1037 * np.arange(n) / SR)
    f, amp = al.amplitude_spectrum(x, SR)
    print(f"durata {1000*n/SR:.2f} ms | passo bin {SR/n:.2f} Hz | picco {f[np.argmax(amp)]:.2f} Hz")
n = int(0.02 * SR)
x = np.sin(2 * np.pi * 1037 * np.arange(n) / SR)
fig = go.Figure()
for n_fft in (n, 8 * n):
    f, amp = al.amplitude_spectrum(x, SR, n_fft=n_fft)
    fig.add_trace(go.Scatter(x=f, y=amp, mode="lines+markers", line=dict(width=1),
                             marker=dict(size=3),
                             name=f"FFT {n_fft}: passo {SR / n_fft:.2f} Hz"))
fig.add_vline(x=1037, line=dict(color="gray", dash="dash"), annotation_text="1037 Hz")
fig.update_xaxes(range=[800, 1250], title_text="frequenza [Hz]")
fig.update_yaxes(title_text="ampiezza")
fig.update_layout(height=340, title="Stessi campioni, diversa densità della griglia")
fig.show()
